# Iran county names & boundaries overlay for model-output maps

Adds to a raw model-output map (e.g. Meteologix ECMWF precipitation):
county names, county / province / international boundaries, coastlines, island names,
sea, strait and neighbouring-country names, and a credit line under the image.

* **Slanted county names:** each name follows the long axis of its county (PCA of the county
  shape) when that fits at least as large as a horizontal name.
* **Area labels (regional frames):** each county name is placed at the most central point of its
  county where the whole label fits inside without touching any boundary; the font shrinks or the
  name wraps onto two lines when needed. Counties created after the boundary data (2023) share a
  polygon with their parent county; the polygon is split between the county seats only for label
  placement (no boundary is invented).
* **Point labels (optional):** county seats with hollow squares, collision-avoiding labels.
* **Province capitals:** red star at the city plus a large black name with a white outline, all at the
  same size. Whole-country frames show only the county names that fit at a readable size.
* **Model numbers** (contour labels, maximum) are detected and never covered by any label.
* **Borders:** Iran's land border is the strongest line; other countries' borders are grey dashed.
* **Seas:** written along the body of water (slanted, stretched with kashida) with a white outline.
* **Base map:** enlarged without changing any value. The precipitation colour classes get smooth edges,
  but the centre of every original pixel keeps exactly its original colour (checked and reported),
  so no class, maximum or contour value is altered. Contour numbers and isolines are only enlarged.
* **Font:** B Yekan (installed copy, or the MIT-licensed Yekan package downloaded once); Vazirmatn as fallback.
* **Data:** official 1404 county list (`sajaddp/list-of-cities-in-Iran`), geoBoundaries IRN ADM1/ADM2
  (OpenStreetMap 2023, ODbL), Natural Earth 1:10m coastlines / borders. All pinned and cached.
* **Output:** high-resolution PNG + PDF (labels and lines are vector in the PDF).

In [1]:
# ============================================================
# 0) Install missing packages (run once if needed)
# ============================================================
# !pip install matplotlib pandas pillow scipy arabic-reshaper python-bidi

In [2]:
# ============================================================
# 1) Configuration
# ============================================================
from pathlib import Path

# ############################################################################
# ##  CHANGE FOR EACH MAP  ###################################################
# ############################################################################
INPUT_IMAGE  = "xx_model-en-999-7-xz_modez_2026110600_186_15552_157.png"          # clean Meteologix map (no lines, no names)
OUTPUT_IMAGE = "ecmwf-final.png"      # high-resolution PNG result
OUTPUT_PDF   = "map_labeled.pdf"      # PDF result (None = no PDF)

# Copy these from the bottom of the Meteologix map:
#   model name ("JMA from ...")  and the period "From ... UTC  to ... UTC"
MODEL     = "اروپا (ECMWF)"                     # e.g. JMA, ECMWF, ICON, GFS
START_UTC = "2026-11-06 00:00"        # "From Fri 2026-11-06, 06:00 UTC"  ->  "2026-11-06 06:00"
END_UTC   = "2026-11-13 18:00"        # "to Wed 2026-11-11, 18:00 UTC"    ->  "2026-11-11 18:00"
SHOW_HEADER = True                    # False = no title box above the map
# ############################################################################

CACHE_DIR    = Path("map_label_data")      # cached CSV / GeoJSON / font files

# --- Frame presets ------------------------------------------------------------
#   size        : image size in pixels (first test for automatic selection)
#   signature   : fingerprint of the region-name text in the footer (separates frames of equal size)
#   projection  : "mercator" or "platecarree" (an affine fit handles different x/y scales)
#   gcps        : ground-control points (pixel x, pixel y, lon, lat); origin = top-left corner
#   map_box     : (x0, y0, x1, y1) pixel area of the map itself
#   label_mode  : "area" (names inside counties) or "point" (names next to county seats)
# Neighbouring countries (name, lon, lat): the label goes to the nearest free spot
NEIGHBOURS = [("ترکیه", 43.75, 38.55), ("عراق", 44.6, 33.2), ("عربستان سعودی", 46.6, 26.6),
              ("کویت", 47.6, 29.35), ("ترکمنستان", 58.6, 38.5), ("افغانستان", 63.0, 33.3),
              ("پاکستان", 63.7, 27.4), ("امارات", 54.6, 24.95), ("قطر", 51.2, 25.35)]

FRAME_PRESETS = {
    "meteologix_bushehr": dict(
        size=(760, 760), projection="platecarree", label_mode="area",
        signature="00000000000000000000000000000000000000000000000000000000000001f800020004000000000000000000018800020004000000000000000000018d19e3e1c7c7800000000000000001891923236646000000000000000001f919822224440000000000000000018d18e233f4640000000000000000018d18323204640000000000000000018d9932332464000000000000000001f8f9e231c4640000000000000000000000000000000000000000000000",
        # Fitted to 13 city markers of the map (RMS 1.6 px), then refined on the coastline and the
        # Bushehr-Fars border: x = -6818.47 + 139.7576*lon ;  y = 4903.07 - 159.5903*lat
        gcps=[(29.66, 594.13, 49, 27), (728.44, 594.13, 54, 27),
              (29.66, 115.36, 49, 30), (728.44, 115.36, 54, 30), (379.05, 354.74, 51.5, 28.5)],
        map_box=(0, 2, 760, 615),
        focus_province="بوشهر",                          # counties of this province are emphasised
        # OSM coast for Iran: full detail for this province, simplified elsewhere
        coast_source="osm", osm_coast_adm1="Bushehr",
        seas=[("خلیج فارس", (49.25, 28.85), (51.20, 27.35), 28)],
        countries=[("عربستان سعودی", 49.25, 27.15)],
        upscale=6, focus_sizes=[8.5, 7.8, 7.0, 6.3, 5.6, 5.0], county_sizes=[5.2, 4.7, 4.2, 3.7, 3.3],
        leader_lines=True, capital_size=15, capital_symbol=9,
        island_fontsize=5.0, country_fontsize=8, credit_fontsize=5.5, note_fontsize=3.5,
    ),
    "meteologix_fars": dict(
        size=(760, 760), projection="platecarree", label_mode="area",
        signature="00000000000000000000000000000000000000000000000000000000000001f8000000000000000000000000000180000000000000000000000000000181c3de000000000000000000000001806312000000000000000000000001e0221800000000000000000000000183e20e00000000000000000000000182220300000000000000000000000182621300000000000000000000000183e21e0000000000000000000000000000000000000000000000000000",
        # Fitted to the map's own boundary lines (median misfit 0.3 px):
        # x = -4399.12 + 90.0558*lon ;  y = 3343.08 - 103.1589*lat   (equirectangular, std. parallel ~29.2N)
        gcps=[(103.67, 557.79, 50.0, 27.0), (644.00, 557.79, 56.0, 27.0),
              (103.67, 41.99, 50.0, 32.0), (644.00, 41.99, 56.0, 32.0), (373.83, 299.89, 53.0, 29.5)],
        map_box=(0, 2, 760, 615),
        # Seas: (name, (lon, lat) start, (lon, lat) end, max size pt). With two points the name is written
        # along that line (slanted, stretched with kashida); with None it is written horizontally.
        seas=[("خلیج فارس", (50.35, 28.35), (52.55, 26.75), 40),
              ("تنگه هرمز", (56.68, 26.63), None, 7)],
        countries=[("عربستان سعودی", 49.35, 26.80)],
        upscale=6, county_sizes=[4.6, 4.2, 3.8, 3.4, 3.0, 2.7], leader_lines=True,
        capital_size=12.8, capital_symbol=7.5,
        island_fontsize=3.8, country_fontsize=6.5, credit_fontsize=5.5, note_fontsize=3.5,
    ),
    "meteologix_iran": dict(
        size=(760, 760), projection="mercator", label_mode="area",
        signature="00000000000000000000000000000000000000000000000000000000000001800000000000000000000000000001800000000000000000000000000001bc71f0000000000000000000000001b01990000000000000000000000001a00910000000000000000000000001a0f918000000000000000000000001a08918000000000000000000000001a09918000000000000000000000001a0f9180000000000000000000000000000000000000000000000000000",
        # Fitted to Natural Earth coastlines + borders (median misfit 0.9 px):
        # x = -1503.97 + 35.039*lon ;  y = 1502.70 - 35.039*mercator_y(lat)
        gcps=[(72.78, 558.70, 45.0, 26.0), (598.37, 558.70, 60.0, 26.0),
              (72.78, 149.03, 45.0, 36.0), (598.37, 149.03, 60.0, 36.0), (335.57, 359.25, 52.5, 31.0)],
        map_box=(0, 2, 760, 615),
        seas=[("دریای کاسپین", (49.75, 38.45), (53.35, 38.25), 16),
              ("خلیج فارس", (49.75, 28.45), (54.10, 26.30), 20),
              ("دریای عمان", (57.55, 25.12), (60.90, 24.92), 12)],
        countries=NEIGHBOURS,
        # whole country: only county names that fit at a readable size, no leader lines
        upscale=6, county_sizes=[2.8, 2.5, 2.25], leader_lines=False,
        county_style=dict(color="#3a3a3a", linewidth=0.35, alpha=0.6),
        capital_size=7.5, capital_symbol=4.5,
        island_fontsize=2.2, country_fontsize=5, credit_fontsize=5.5, note_fontsize=3.5,
    ),
    "icon13_aryatafkik": dict(
        size=(1280, 1173), projection="platecarree", label_mode="area", signature=None,
        gcps=[(430.3, 1000.5, 50.0, 25.0), (723.6, 1000.5, 55.0, 25.0), (1016.8, 1000.5, 60.0, 25.0),
              (723.6, 706.7, 55.0, 30.0), (1016.8, 706.7, 60.0, 30.0)],
        map_box=(0, 44, 1280, 1088),
        seas=[("دریای کاسپین", (50.2, 39.4), (52.9, 38.6), 26),
              ("خلیج فارس", (49.6, 28.6), (54.2, 26.2), 32),
              ("دریای عمان", (57.9, 24.6), (61.3, 24.1), 20)],
        countries=NEIGHBOURS,
        upscale=4, county_sizes=[3.8, 3.4, 3.0], leader_lines=False,
        county_style=dict(color="#3a3a3a", linewidth=0.4, alpha=0.6),
        capital_size=10.0, capital_symbol=6.0,
        island_fontsize=3.0, country_fontsize=6, credit_fontsize=7, note_fontsize=4.5,
    ),
}
FRAME = "auto"            # "auto" = choose by image size + footer signature, or a preset name

# --- Islands (label anchors; all verified inside Natural Earth island outlines) ---
ISLANDS = [("خارگ", 50.305, 29.246), ("خارکو", 50.346, 29.321), ("شیف", 50.880, 29.030), ("کیش", 53.98, 26.53), ("قشم", 55.85, 26.77),
           ("هرمز", 56.46, 27.06), ("لارک", 56.36, 26.86), ("هنگام", 55.88, 26.64),
           ("لاوان", 53.25, 26.80), ("هندورابی", 53.62, 26.67), ("فارور", 54.52, 26.29),
           ("سیری", 54.54, 25.91), ("ابوموسی", 55.03, 25.87)]
ISLAND_PREFIX = "جزیره "   # an island whose name is already a county label is not repeated

# --- Data corrections ------------------------------------------------------------
# Polygons whose county seat is mis-geocoded in the source list: polygon name -> Persian county name.
# (The bad point of that county is then ignored.) Seats lying outside their own province are dropped
# automatically.
POLYGON_NAME_FIXES = {"Parsian": "پارسیان"}

# --- Lines (widths in points; 1 pt ~ 1.4 input pixels) ------------------------------
# (a preset may override the county style with its own "county_style")
COUNTY_STYLE   = dict(color="#a0461e", linewidth=0.55)                      # county borders
PROVINCE_STYLE = dict(color="#1b1b1b", linewidth=1.15)                      # province borders
IRAN_STYLE     = dict(color="#000000", linewidth=1.9)                       # Iran's land border
FOREIGN_STYLE  = dict(color="#6e6e6e", linewidth=0.7, linestyle=(0, (5, 2.5)))  # other countries
COAST_STYLE    = dict(color="#101010", linewidth=0.6)                       # coastlines + islands
# Coastline smoothing: creeks / channels narrower than about 2 x radius (input px) are closed
COAST_SMOOTH_PX       = 2.5    # Iranian coast outside the focus province
COAST_SMOOTH_FOCUS_PX = 1.5    # coast of the focus province (keeps more shape, still no clutter)
FOCUS_STYLE    = dict(color="#000000", linewidth=2.2)                       # border of the focus province
FOCUS_CASING   = dict(color="white", linewidth=4.2)                         # white casing under it
LABEL_LINE_CLEARANCE = 0.8   # pt, minimum gap between an area label and any boundary line

# --- Labels ---------------------------------------------------------------------------
# --- Font --------------------------------------------------------------------------------
# B Yekan. FONT_FILE = path to a .ttf (e.g. r"C:\Windows\Fonts\BYekan.ttf"); None = use an installed
# Yekan font if found, else download the MIT-licensed Yekan package (npm "typeface-yekan" 1.0.11).
FONT_FILE    = None
FONT_NPM_URL = "https://registry.npmjs.org/typeface-yekan/-/typeface-yekan-1.0.11.tgz"
HEAVY_STROKE = 0.045           # single-weight font: emphasis = outline in the text colour (x font size)
EDGE_MARGIN_PX = 4             # labels keep at least this many input pixels from the map frame
FOCUS_MIN_INSIDE = 6.3         # focus-province names: preferred minimum size inside the county
# Narrow focus counties (e.g. a coastal strip): if the name does not fit at FOCUS_MIN_INSIDE, it is
# written inside the strip along its long axis at one of these smaller sizes, with a tighter gap
# to the border, before it is ever placed outside the county.
NARROW_SIZES     = [6.0, 5.6, 5.2, 4.8, 4.4]

# --- County seats of the focus province (exact location: small circle + city name) -----------
SHOW_FOCUS_SEATS = True
SEAT_SIZES       = [5.5, 5.0, 4.5]  # pt, city-name sizes (first that fits)
SEAT_MARKER      = 3.2         # pt, circle diameter
SEAT_COLOR       = "#1a1a1a"   # city name colour
SEAT_FACE        = "white"     # circle fill
SEAT_EDGE        = "#111111"   # circle outline
COASTAL_SEAT_PX  = 3.0         # a seat within this distance (input px) of the coast is "coastal":
                               # its name goes right next to the circle, over the sea if needed

# --- Header box above the map (None = no header) ---------------------------------------------------
# Model and times come from the "CHANGE FOR EACH MAP" block at the top; they are shown in the
# Persian calendar and Iran time (UTC+03:30).
HEADER = dict(
    model     = MODEL,
    start_utc = START_UTC,
    end_utc   = END_UTC,
    title     = "نقشه پیش‌بینی بارش تجمعی خروجی مدل {model}",
    period    = "از ساعت {t1} {d1} / تا ساعت  {t2} {d2} (به وقت ایران)",
) if SHOW_HEADER else None
HEADER_COLOR    = "#0b2a6f"    # header title colour
HEADER_FILL     = "#f3f6fc"    # header background
IRAN_UTC_OFFSET = (3, 30)
NARROW_CLEARANCE = 0.3         # pt, gap between a narrow-county name and the lines

TEXT_COLOR       = "#111111"   # county and island names (always this colour)
PROTECT_MODEL_NUMBERS = True   # never place any label over the contour numbers / maximum of the map
CAPITAL_STROKE   = 2.0         # pt, white outline around province-capital names
SEA_STROKE       = 0.07        # white outline around sea names, as a fraction of the font size
SEA_KASHIDA_MAX  = 2           # max kashida (ـ) per word used to stretch slanted sea names
SEA_STRETCH      = 0.6         # stretch the sea name with kashida up to this share of its line
BANNER = "auto"                # dark ECMWF banner at the top of the map: "auto", True or False
CAPITAL_COLOR    = "#000000"   # province-capital names (white outline keeps them readable)
CAPITAL_STAR_COLOR = "#d90429" # province-capital star
CAPITAL_MARKER   = "*"         # province-capital symbol (matplotlib marker; "*" = star)
SEA_COLOR        = "#0b2a6f"
COUNTRY_COLOR    = "#5a5a5a"
LIGHT_TEXT_COLOR = "white"
ADAPTIVE_SEA_COLOR = False     # sea names have a white outline, so a fixed colour reads everywhere
SLANTED_COUNTY_NAMES = True    # write county names along the long axis of their county
MAX_SLANT        = 35          # degrees; steeper counties fall back to the nearest allowed angle
# With a focus province, only its counties are slanted; neighbouring counties stay horizontal
WRAP_TWO_WORDS   = True        # allow "بندر عباس" -> two lines when one line does not fit
SHOW_SEATS_AREA  = False       # area mode: also mark county seats with a small square

# --- Base map --------------------------------------------------------------------------
UPSCALE = None                 # None = preset value; PNG = input size x UPSCALE
BASE_UPSCALE_METHOD = "faithful"   # "faithful" (smooth edges, original values kept), "lanczos", "nearest"

# --- Text under the image ------------------------------------------------------------
CREDIT_TEXT = ("Iran county names overlay developed by Hossein Shokoohi, "
               "Bushehr Provincial Meteorology Office")
SOURCE_NOTE = None             # optional second line under the credit (None = no second line)

# --- Data sources (pinned commits) ----------------------------------------------------
DATA_BASE = ("https://raw.githubusercontent.com/sajaddp/list-of-cities-in-Iran/"
             "ace21bb251f96ced350bbc01541acffd71121feb/dist/csv")
GB_BASE   = ("https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/"
             "8ce2700661a28ee944350e2b329bbabb58d2ff83/releaseData/gbOpen/IRN")
NE_BASE   = ("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/"
             "ca96624a56bd078437bca8184e78163e5039ad19/geojson")
FONT_BASE = "https://raw.githubusercontent.com/rastikerdar/vazirmatn/master/fonts/ttf"

In [ ]:
# ============================================================
# 2) Imports, downloads and data loading
# ============================================================
import json
import urllib.request
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle
from matplotlib.path import Path as MplPath
from PIL import Image, ImageDraw
from scipy import ndimage as ndi
import matplotlib.patheffects as pe

CACHE_DIR.mkdir(parents=True, exist_ok=True)

def fetch(url, dest):
    """Download a file once and reuse the cached copy afterwards."""
    dest = Path(dest)
    if not dest.exists() or dest.stat().st_size == 0:
        print(f"Downloading {url}")
        urllib.request.urlretrieve(url, dest)
    return dest

county_csv  = fetch(f"{DATA_BASE}/county-centers.csv",    CACHE_DIR / "county-centers.csv")
capital_csv = fetch(f"{DATA_BASE}/province-capitals.csv", CACHE_DIR / "province-capitals.csv")
adm1_file   = fetch(f"{GB_BASE}/ADM1/geoBoundaries-IRN-ADM1.geojson", CACHE_DIR / "geoBoundaries-IRN-ADM1.geojson")
adm2_file   = fetch(f"{GB_BASE}/ADM2/geoBoundaries-IRN-ADM2.geojson", CACHE_DIR / "geoBoundaries-IRN-ADM2.geojson")
coast_file  = fetch(f"{NE_BASE}/ne_10m_coastline.geojson", CACHE_DIR / "ne_10m_coastline.geojson")
border_file = fetch(f"{NE_BASE}/ne_10m_admin_0_boundary_lines_land.geojson",
                    CACHE_DIR / "ne_10m_admin_0_boundary_lines_land.geojson")
import tarfile
from fontTools.ttLib import TTFont          # installed with matplotlib

PERSIAN_LETTERS = [0x067E, 0x0686, 0x0698, 0x06A9, 0x06AF, 0x06CC]   # پ چ ژ ک گ ی

def font_ok(path):
    try:
        cmap = TTFont(str(path), fontNumber=0).getBestCmap()
        return all(c in cmap for c in PERSIAN_LETTERS)
    except Exception:
        return False

def find_yekan():
    """Path of a usable B Yekan font: FONT_FILE, an installed Yekan, or the downloaded package."""
    if FONT_FILE:
        if font_ok(FONT_FILE):
            return Path(FONT_FILE)
        print(f"WARNING: FONT_FILE {FONT_FILE} is missing or has no Persian letters.")
    installed = [f for f in fm.findSystemFonts()
                 if "yekan" in Path(f).stem.lower().replace(" ", "") and "bakh" not in Path(f).stem.lower()]
    installed.sort(key=lambda f: ("byekan" not in Path(f).stem.lower().replace(" ", ""), len(f)))
    for f in installed:
        if font_ok(f):
            return Path(f)
    dest = CACHE_DIR / "Yekan.ttf"
    if not dest.exists():
        try:
            tgz = fetch(FONT_NPM_URL, CACHE_DIR / "typeface-yekan.tgz")
            with tarfile.open(tgz) as tf:
                member = tf.getmember("package/dist/Yekan.ttf")      # read one known file only
                dest.write_bytes(tf.extractfile(member).read())
        except Exception as e:
            print(f"WARNING: could not get the Yekan font ({e}).")
            return None
    return dest if font_ok(dest) else None

yekan = find_yekan()
if yekan is not None:
    FONT_PATH = yekan
    SINGLE_WEIGHT = True                    # Yekan has one weight; emphasis via outline
else:                                       # fallback: Vazirmatn (three weights)
    print("Using Vazirmatn instead of B Yekan.")
    FONT_PATH = fetch(f"{FONT_BASE}/Vazirmatn-Bold.ttf", CACHE_DIR / "Vazirmatn-Bold.ttf")
    SINGLE_WEIGHT = False
def font_props(path):
    """FontProperties for `path` with DejaVu Sans as fallback for any glyph the font lacks
    (e.g. some Latin letters in older "B" fonts), so no glyph is ever missing."""
    fm.fontManager.addfont(str(path))
    name = fm.FontProperties(fname=str(path)).get_name()
    return fm.FontProperties(family=[name, "DejaVu Sans"])

FP = {}
for weight in ("Regular", "Bold", "Black"):
    if SINGLE_WEIGHT:
        FP[weight] = font_props(FONT_PATH)
    else:
        f = fetch(f"{FONT_BASE}/Vazirmatn-{weight}.ttf", CACHE_DIR / f"Vazirmatn-{weight}.ttf")
        FP[weight] = font_props(f)
_tt = TTFont(str(FONT_PATH), fontNumber=0)
FONT_CMAP = set(_tt.getBestCmap())
FONT_HAS_GSUB = "GSUB" in _tt
print(f"Font: {FONT_PATH.name}  (OpenType shaping table: {FONT_HAS_GSUB})")

def read_geojson(path, bbox=(35, 15, 75, 46)):
    """Features as (name, [outer rings or lines as (N, 2) lon/lat arrays]) inside bbox."""
    out = []
    for f in json.load(open(path, encoding="utf-8"))["features"]:
        g = f["geometry"]
        if g is None:
            continue
        t, c = g["type"], g["coordinates"]
        if t == "Polygon":
            parts = [c[0]]
        elif t == "MultiPolygon":
            parts = [poly[0] for poly in c]
        elif t == "LineString":
            parts = [c]
        elif t == "MultiLineString":
            parts = c
        else:
            continue
        arrs = []
        for p in parts:
            a = np.asarray(p, dtype=float)[:, :2]
            if (a[:, 0].max() >= bbox[0] and a[:, 0].min() <= bbox[2] and
                    a[:, 1].max() >= bbox[1] and a[:, 1].min() <= bbox[3]):
                arrs.append(a)
        if arrs:
            p = f["properties"]
            out.append((p.get("shapeName") or "{}|{}".format(p.get("ADM0_A3_L"), p.get("ADM0_A3_R")), arrs))
    return out

adm1   = read_geojson(adm1_file)                     # provinces
adm2   = read_geojson(adm2_file)                     # counties (432 polygons, 2023)
coast  = [a for _, arrs in read_geojson(coast_file) for a in arrs]
_borders = read_geojson(border_file)
iran_border    = [a for n, arrs in _borders if "IRN" in n.split("|") for a in arrs]
foreign_border = [a for n, arrs in _borders if "IRN" not in n.split("|") for a in arrs]

# --- County seats: load and validate -------------------------------------------------
counties = pd.read_csv(county_csv, encoding="utf-8").dropna(subset=["latitude", "longitude"])
capitals = pd.read_csv(capital_csv, encoding="utf-8")
cap_keys = set(zip(capitals["center_name"], capitals["province_id"]))
counties["is_capital"] = [(c, p) in cap_keys for c, p in zip(counties["center_name"], counties["province_id"])]

def poly_index(features, pts):
    """For every point, the index of the feature polygon containing it (-1 if none)."""
    idx = np.full(len(pts), -1)
    for k, (_, rings) in enumerate(features):
        inside = np.zeros(len(pts), bool)
        for r in rings:
            inside |= MplPath(r).contains_points(pts)
        idx[inside & (idx < 0)] = k
    return idx

pts = counties[["longitude", "latitude"]].to_numpy()
counties["adm1"] = poly_index(adm1, pts)
counties["adm2"] = poly_index(adm2, pts)
# A seat lying in another province than most seats of its province is mis-geocoded -> dropped
major = {pid: Counter(g["adm1"]).most_common(1)[0][0] for pid, g in counties.groupby("province_id")}
bad = counties.apply(lambda r: r["adm1"] != major[r["province_id"]], axis=1)
# Keep seats that are only slightly outside (border towns): within 0.25 deg of their province
for i in counties.index[bad]:
    if major[counties.at[i, "province_id"]] < 0:
        continue
    rings = adm1[major[counties.at[i, "province_id"]]][1]
    d = min(np.hypot(r[:, 0] - counties.at[i, "longitude"], r[:, 1] - counties.at[i, "latitude"]).min()
            for r in rings)
    if d < 0.25:                     # border towns stay; seats far away are clearly wrong
        bad[i] = False
print("Dropped mis-geocoded seats:", list(counties.loc[bad, "county_name"]))
counties = counties[~bad]
fixed_names = set(POLYGON_NAME_FIXES.values())
counties.loc[counties["county_name"].isin(fixed_names), "adm2"] = -2   # handled by polygon fix
counties = counties.reset_index(drop=True)
print(f"County seats: {len(counties)}  |  polygons: {len(adm2)} counties, {len(adm1)} provinces")

In [ ]:
# ============================================================
# 3) Frame selection and georeferencing
# ============================================================
img = Image.open(INPUT_IMAGE).convert("RGB")
W, H = img.size
print(f"Input image: {W} x {H} px")

def footer_signature(im):
    """Fingerprint of the region name in the footer (e.g. "Iran", "Fars")."""
    g = im.convert("L").crop((10, 724, 130, 736)).resize((120, 12), Image.BOX)
    return np.packbits((np.asarray(g) < 150).astype(np.uint8).ravel()).tobytes().hex()

def hamming(a, b):
    return bin(int(a, 16) ^ int(b, 16)).count("1")

if FRAME == "auto":
    same_size = [k for k, p in FRAME_PRESETS.items() if tuple(p["size"]) == (W, H)]
    if not same_size:
        raise ValueError(f"No preset for a {W} x {H} image; set FRAME or add a preset.")
    if len(same_size) > 1:
        sig = footer_signature(img)
        dists = {k: hamming(sig, FRAME_PRESETS[k]["signature"]) for k in same_size
                 if FRAME_PRESETS[k]["signature"]}
        frame_name = min(dists, key=dists.get)
        print("Footer signature distances:", dists)
        if dists[frame_name] > 30:
            print("WARNING: no preset matches this footer well; set FRAME explicitly.")
    else:
        frame_name = same_size[0]
else:
    frame_name = FRAME
P = FRAME_PRESETS[frame_name]
ups = UPSCALE or P["upscale"]
print(f"Frame preset: {frame_name}  ({P['projection']}, labels: {P['label_mode']})")

gcps = np.array(P["gcps"], dtype=float)
box = np.array(P["map_box"], dtype=float)

# ECMWF maps carry a semi-transparent dark banner over the top ~21 px of the map
def has_banner(im):
    a = np.asarray(im.convert("L"), dtype=float)
    r = (a[12] + 1) / (a[23] + 1)
    return ((r > 0.35) & (r < 0.65)).mean() > 0.25
banner = has_banner(img) if BANNER == "auto" else bool(BANNER)
if banner and box[1] < 21:
    box[1] = 21
print(f"Top banner: {banner}  |  map area: {box.astype(int).tolist()}")

def project(lon, lat):
    lon = np.asarray(lon, dtype=float); lat = np.asarray(lat, dtype=float)
    if P["projection"] == "mercator":
        return lon, np.degrees(np.log(np.tan(np.pi / 4 + np.radians(lat) / 2)))
    return lon, lat

GX, GY = project(gcps[:, 2], gcps[:, 3])
A = np.column_stack([np.ones(len(gcps)), GX, GY])
coef_x, *_ = np.linalg.lstsq(A, gcps[:, 0], rcond=None)
coef_y, *_ = np.linalg.lstsq(A, gcps[:, 1], rcond=None)

def lonlat_to_px(lon, lat):
    X, Y = project(lon, lat)
    return (coef_x[0] + coef_x[1] * X + coef_x[2] * Y, coef_y[0] + coef_y[1] * X + coef_y[2] * Y)

res = np.hypot(*(np.array(lonlat_to_px(gcps[:, 2], gcps[:, 3])) - gcps[:, :2].T))
print(f"Scale: {coef_x[1]:.2f} px/deg (x)  |  GCP residuals: max {res.max():.2f} px")

counties["x"], counties["y"] = lonlat_to_px(counties["longitude"], counties["latitude"])
in_box = lambda x, y: (x >= box[0]) & (x < box[2]) & (y >= box[1]) & (y < box[3])
print(f"County seats inside the map: {in_box(counties['x'], counties['y']).sum()}")

In [ ]:
# ============================================================
# 4) Base map enlargement that keeps every original value
# ============================================================
def faithful_upscale(rgb, f, min_count=40, open_ratio=0.45, field_sigma=0.6):
    """Enlarge a flat-colour weather map by an integer factor f without changing its values.

    * Area colours (precipitation classes) get smooth edges: each class mask is slightly smoothed,
      enlarged, and the strongest class wins - but only within the outer quarter of each original
      pixel; the central half always keeps the pixel's own class.
    * Non-area pixels (contour numbers, isolines, anti-aliasing) and their neighbours use plain
      Lanczos enlargement of the original image.
    * Finally the centre of every original pixel is set to its exact original colour.
    Returns the enlarged image and the share of original pixels reproduced exactly at their centre.
    """
    h, w, _ = rgb.shape
    H2, W2 = h * f, w * f
    key = (rgb[..., 0].astype(np.int32) << 16) | (rgb[..., 1].astype(np.int32) << 8) | rgb[..., 2]
    vals, inv, counts = np.unique(key.ravel(), return_inverse=True, return_counts=True)
    inv = inv.reshape(h, w)
    fill_ids = []
    for i in np.argsort(-counts):
        if counts[i] < min_count or len(fill_ids) == 255:
            break
        if ndi.binary_opening(inv == i, structure=np.ones((3, 3), bool)).sum() / counts[i] >= open_ratio:
            fill_ids.append(i)
    fill_ids = np.array(fill_ids)
    lut = np.full(len(vals), 255, np.uint8)
    lut[fill_ids] = np.arange(len(fill_ids))
    is_fill = lut[inv] != 255
    _, (iy, ix) = ndi.distance_transform_edt(~is_fill, return_indices=True)
    field_k = lut[inv[iy, ix]]

    best = np.full((H2, W2), -1.0, np.float32)
    win = np.zeros((H2, W2), np.uint8)
    for k in range(len(fill_ids)):
        m = ndi.gaussian_filter((field_k == k).astype(np.float32), field_sigma)
        up = np.asarray(Image.fromarray(m, mode="F").resize((W2, H2), Image.BILINEAR))
        better = up > best
        best[better] = up[better]
        win[better] = k
    del best
    pos = (np.arange(f) + 0.5) / f
    core1 = np.abs(pos - 0.5) <= 0.25
    core = np.tile(np.outer(core1, core1), (h, w))
    win[core] = np.repeat(np.repeat(field_k, f, 0), f, 1)[core]
    del core
    fv = vals[fill_ids]
    out = np.stack([(fv >> 16) & 255, (fv >> 8) & 255, fv & 255], 1).astype(np.uint8)[win]
    del win

    nf = ndi.binary_dilation(~is_fill, structure=np.ones((3, 3), bool))
    if nf.any():
        a = ndi.gaussian_filter(np.repeat(np.repeat(nf.astype(np.float32), f, 0), f, 1), f * 0.35)
        lz = np.asarray(Image.fromarray(rgb).resize((W2, H2), Image.LANCZOS))
        for r0 in range(0, H2, 512):
            r1 = min(H2, r0 + 512)
            al = a[r0:r1, :, None]
            out[r0:r1] = (out[r0:r1] * (1 - al) + lz[r0:r1] * al + 0.5).astype(np.uint8)

    centre = np.where(np.abs(pos - 0.5) <= 0.17)[0]
    for r in centre:
        for c in centre:
            out[r::f, c::f] = rgb
    exact = np.all(out[f // 2::f, f // 2::f] == rgb, axis=2).mean()
    return out, exact

resample = Image.NEAREST if BASE_UPSCALE_METHOD == "nearest" else Image.LANCZOS
base_hi = np.asarray(img.resize((W * ups, H * ups), resample)).copy()
if BASE_UPSCALE_METHOD == "faithful":
    bx0, by0, bx1, by1 = [int(round(v)) for v in box]
    hi, exact = faithful_upscale(np.asarray(img)[by0:by1, bx0:bx1], ups)
    base_hi[by0 * ups:by1 * ups, bx0 * ups:bx1 * ups] = hi
    print(f"Base map enlarged x{ups} ({base_hi.shape[1]} x {base_hi.shape[0]} px); "
          f"original pixels reproduced exactly at their centre: {exact * 100:.3f}%")
else:
    print(f"Base map enlarged x{ups} with {BASE_UPSCALE_METHOD}")

In [ ]:
# ============================================================
# 5) Persian text shaping
# ============================================================
# Matplotlib >= 3.11 (libraqm) shapes Arabic-script text itself; older versions need
# arabic_reshaper + python-bidi. Applying both breaks the text, so detect which case applies
# (tested with the default DejaVu Sans, which has full OpenType Arabic shaping).
import unicodedata

def needs_manual_shaping():
    try:
        import arabic_reshaper
    except ImportError:
        return False
    fig = plt.figure(figsize=(2, 1), dpi=100)
    r = fig.canvas.get_renderer()
    def width(s):
        t = fig.text(0, 0, s, family="DejaVu Sans", fontsize=20)
        w = t.get_window_extent(r).width
        t.remove()
        return w
    raw = "بببببب"
    result = width(raw) > 1.15 * width(arabic_reshaper.reshape(raw))
    plt.close(fig)
    return result

NATIVE_SHAPING = not needs_manual_shaping()
# Pre-shaped letters (presentation forms) are needed when matplotlib cannot shape, or when the font
# has no OpenType shaping table (many older "B" fonts)
PRESHAPE = (not NATIVE_SHAPING) or (not FONT_HAS_GSUB)
if PRESHAPE:
    import arabic_reshaper
    from bidi.algorithm import get_display

# Some fonts lack a few presentation forms (e.g. Farsi yeh / keheh / gaf forms): use an equivalent
ALT_BASE = {0x06CC: [0x064A], 0x06A9: [0x0643]}          # Farsi yeh -> Arabic yeh, keheh -> kaf
def _remap(ch):
    o = ord(ch)
    if o in FONT_CMAP or not (0xFB50 <= o <= 0xFEFF):
        return ch
    dec = unicodedata.decomposition(ch).split()
    if len(dec) != 2:
        return ch
    form, base = dec[0].strip("<>"), int(dec[1], 16)
    if base == 0x06CC and form in ("final", "isolated"):
        alts = [(0x0649, form)]                           # dotless yeh (alef maksura) shapes
    else:
        alts = [(a, form) for a in ALT_BASE.get(base, [])]
    for a, fo in alts:
        try:
            c2 = unicodedata.lookup(f"{unicodedata.name(chr(a))} {fo.upper()} FORM")
            if ord(c2) in FONT_CMAP:
                return c2
        except KeyError:
            pass
    return chr(base) if base in FONT_CMAP else ch

_reported = set()
def _check_glyphs(original, text):
    """Print (once per label) characters the main font lacks; they are drawn from DejaVu Sans."""
    miss = sorted({c for c in text if ord(c) not in FONT_CMAP and not c.isspace()})
    if miss and original not in _reported:
        _reported.add(original)
        print(f"  note: label {original!r} uses characters missing from {FONT_PATH.name}: "
              f"{[f'U+{ord(c):04X}' for c in miss]} (drawn with DejaVu Sans)")

def fa(s):
    """Persian string ready for matplotlib (line by line)."""
    r = _fa(s)
    _check_glyphs(str(s), r)
    return r

def _fa(s):
    if not PRESHAPE:
        return str(s)
    out = []
    for line in str(s).split("\n"):
        shaped = "".join(_remap(c) for c in arabic_reshaper.reshape(line))
        # matplotlib >= 3.11 still applies the bidi algorithm itself; older versions need it here
        out.append(shaped if NATIVE_SHAPING else get_display(shaped))
    return "\n".join(out)

print(f"matplotlib {matplotlib.__version__}  |  native shaping: {NATIVE_SHAPING}  |  "
      f"pre-shaped letters: {PRESHAPE}")

In [ ]:
# ============================================================
# 6) Boundary lines in pixel space
# ============================================================
S = 4                                   # label-grid cells per input pixel
GW, GH = W * S, H * S
PT = 100 / 72                           # input pixels per point (figure is 100 px per inch)

def to_px(a):
    x, y = lonlat_to_px(a[:, 0], a[:, 1])
    return np.column_stack([x, y])

def rasterize_ids(features):
    """Grid (GH x GW) with the 1-based index of the polygon covering each cell (0 = none)."""
    im = Image.new("I", (GW, GH), 0)
    d = ImageDraw.Draw(im)
    for k, (_, rings) in enumerate(features):
        for r in rings:
            p = to_px(r) * S
            if p[:, 0].max() < 0 or p[:, 0].min() > GW or p[:, 1].max() < 0 or p[:, 1].min() > GH:
                continue
            d.polygon([tuple(v) for v in p], fill=k + 1)
    return np.asarray(im, dtype=np.int32)

county_grid = rasterize_ids(adm2)

def split_runs(p, keep, min_len=4.0):
    """Split polyline p into runs of consecutive kept segments; drop runs shorter than min_len px
    (tiny fragments along jagged coasts would otherwise show up as dots)."""
    out, run = [], [p[0]]
    def flush(r):
        if len(r) > 1:
            a = np.array(r)
            if np.hypot(*np.diff(a, axis=0).T).sum() >= min_len:
                out.append(a)
    for i, k in enumerate(keep):
        if k:
            run.append(p[i + 1])
        else:
            flush(run)
            run = [p[i + 1]]
    flush(run)
    return out

def interior_lines(features):
    """Edges shared by two polygons (= internal borders). Coastline and the outer border belong to
    a single polygon and are left to Natural Earth, so creeks and mangrove channels are not drawn."""
    def segs(r):
        a = np.round(r, 6)
        u, v = a[:-1], a[1:]
        swap = (u[:, 0] > v[:, 0]) | ((u[:, 0] == v[:, 0]) & (u[:, 1] > v[:, 1]))
        lo = np.where(swap[:, None], v, u); hi = np.where(swap[:, None], u, v)
        return list(zip(map(tuple, lo), map(tuple, hi)))
    count = Counter()
    for _, rings in features:
        count.update(set(sg for r in rings for sg in segs(r)))
    out, drawn = [], set()
    for _, rings in features:
        for r in rings:
            p = to_px(r)
            if (p[:, 0].max() < box[0] or p[:, 0].min() > box[2] or
                    p[:, 1].max() < box[1] or p[:, 1].min() > box[3]) or len(p) < 2:
                continue
            ss = segs(r)
            keep = np.array([count[sg] >= 2 and sg not in drawn for sg in ss])
            drawn.update(sg for sg, k in zip(ss, keep) if k)
            out += split_runs(p, keep, min_len=0.0)
    return out

def plain_lines(arrs):
    out = []
    for a in arrs:
        p = to_px(a)
        if (p[:, 0].max() >= box[0] and p[:, 0].min() <= box[2] and
                p[:, 1].max() >= box[1] and p[:, 1].min() <= box[3]) and len(p) > 1:
            out.append(p)
    return out

def edge_count(features):
    """Number of polygons sharing each (rounded) edge."""
    def segs(r):
        a = np.round(r, 6)
        u, v = a[:-1], a[1:]
        swap = (u[:, 0] > v[:, 0]) | ((u[:, 0] == v[:, 0]) & (u[:, 1] > v[:, 1]))
        lo = np.where(swap[:, None], v, u); hi = np.where(swap[:, None], u, v)
        return list(zip(map(tuple, lo), map(tuple, hi)))
    cnt = Counter()
    for _, rings in features:
        cnt.update(set(sg for r in rings for sg in segs(r)))
    return cnt, segs

def select_edges(rings, keep_fn, segs, min_len=0.0):
    out = []
    for r in rings:
        p = to_px(r)
        if (p[:, 0].max() < box[0] or p[:, 0].min() > box[2] or
                p[:, 1].max() < box[1] or p[:, 1].min() > box[3]) or len(p) < 2:
            continue
        out += split_runs(p, np.array([keep_fn(sg) for sg in segs(r)]), min_len=min_len)
    return out

adm1_count, _segs = edge_count(adm1)
coast_lines_px = plain_lines(coast)
if P.get("coast_source") == "osm":
    # Iran's coast = outer edges of the province polygons that are not on a land border;
    # other countries keep the Natural Earth coast (Iran's part of it is removed)
    near_border = np.zeros((GH, GW), bool)
    if iran_border:
        im_ = Image.new("1", (GW, GH), 0); d_ = ImageDraw.Draw(im_)
        for p in plain_lines(iran_border):
            d_.line([tuple(v) for v in p * S], fill=1, width=int(4 * S))
        near_border = np.asarray(im_, dtype=bool)
    def is_coast(sg):
        if adm1_count[sg] != 1:
            return False
        x, y = lonlat_to_px((sg[0][0] + sg[1][0]) / 2, (sg[0][1] + sg[1][1]) / 2)
        gx, gy = int(x * S), int(y * S)
        return not (0 <= gx < GW and 0 <= gy < GH and near_border[gy, gx])
    def simplify(p, tol):
        """Douglas-Peucker line simplification (tolerance in pixels)."""
        if len(p) < 3:
            return p
        a, b = p[0], p[-1]
        ab = b - a
        n = np.hypot(*ab)
        d = (np.abs(ab[0] * (p[:, 1] - a[1]) - ab[1] * (p[:, 0] - a[0])) / n if n > 0
             else np.hypot(p[:, 0] - a[0], p[:, 1] - a[1]))
        i = int(np.argmax(d))
        if d[i] <= tol:
            return np.array([a, b])
        return np.vstack([simplify(p[:i + 1], tol)[:-1], simplify(p[i:], tol)])
    # Iranian coast traced from the county polygons: creeks / channels narrower than ~2 x radius are
    # closed and the line lightly smoothed -> correct position without the jagged OSM detail.
    # The focus province uses a smaller radius (keeps more of its shape).
    import contourpy
    land_all = county_grid > 0
    def smooth_land(rad_px):
        rad = max(1, int(rad_px * S))
        disk = np.hypot(*np.mgrid[-rad:rad + 1, -rad:rad + 1]) <= rad
        closed = ndi.binary_closing(np.pad(land_all, rad, mode="edge"), structure=disk)[rad:-rad, rad:-rad]
        return ndi.gaussian_filter(closed.astype(np.float32), 0.6 * S)
    focus_k = [i for i, (n, _) in enumerate(adm1) if n == P.get("osm_coast_adm1")]
    focus_land = np.zeros_like(land_all)
    for k_ in focus_k:
        im_ = Image.new("1", (GW, GH), 0); d_ = ImageDraw.Draw(im_)
        for r in adm1[k_][1]:
            d_.polygon([tuple(v) for v in to_px(r) * S], fill=1)
        focus_land |= np.asarray(im_, dtype=bool)
    focus_zone = ndi.binary_dilation(focus_land, iterations=int(3 * S))
    iran_coast = []
    for rad_px, zone in ((COAST_SMOOTH_FOCUS_PX, focus_zone), (COAST_SMOOTH_PX, ~focus_zone)):
        if not zone.any():
            continue
        for ln in contourpy.contour_generator(z=smooth_land(rad_px)).lines(0.5):
            if len(ln) < 2:
                continue
            gx_, gy_ = np.clip(ln[:, 0].astype(int), 0, GW - 1), np.clip(ln[:, 1].astype(int), 0, GH - 1)
            keep_pt = zone[gy_, gx_] & ~near_border[gy_, gx_]
            iran_coast += split_runs(ln / S, keep_pt[:-1] & keep_pt[1:], min_len=3.0)
    land = ndi.binary_dilation(land_all, iterations=int(3 * S))
    def off_iran(p):
        g = (p * S).astype(int)
        ok = (g[:, 0] >= 0) & (g[:, 0] < GW) & (g[:, 1] >= 0) & (g[:, 1] < GH)
        inside = np.zeros(len(p), bool)
        inside[ok] = land[g[ok, 1], g[ok, 0]]
        return split_runs(p, ~(inside[:-1] & inside[1:]), min_len=1.0)
    coast_lines_px = iran_coast + [q for p in coast_lines_px for q in off_iran(p)]

focus_lines = []
if P.get("focus_adm1"):
    k = [i for i, (n, _) in enumerate(adm1) if n == P["focus_adm1"]]
    if k:
        focus_lines = select_edges(adm1[k[0]][1], lambda sg: adm1_count[sg] >= 2, _segs)

LINES = {
    "county":   (interior_lines(adm2), P.get("county_style", COUNTY_STYLE), 3.0),
    "foreign":  (plain_lines(foreign_border), FOREIGN_STYLE, 3.2),
    "province": (interior_lines(adm1), PROVINCE_STYLE, 3.4),
    "coast":    (coast_lines_px, COAST_STYLE, 3.6),
    "iran":     (plain_lines(iran_border), IRAN_STYLE, 3.7),
    "focus_casing": (focus_lines, FOCUS_CASING, 3.75),
    "focus":    (focus_lines, FOCUS_STYLE, 3.8),
}
print("Polylines:", {k: len(v[0]) for k, v in LINES.items()})

# Occupancy grid of all lines (+ clearance) for label placement
def line_mask(keys, clearance=None):
    clearance = LABEL_LINE_CLEARANCE if clearance is None else clearance
    im_ = Image.new("L", (GW, GH), 0)
    d_ = ImageDraw.Draw(im_)
    for k in keys:
        lines, style, _ = LINES[k]
        wpx = max(1, int(round((style["linewidth"] + 2 * clearance) * PT * S)))
        for p in lines:
            d_.line([tuple(v) for v in p * S], fill=255, width=wpx, joint="curve")
    return np.asarray(im_) > 0
occ_major = line_mask(["province", "iran", "foreign", "coast", "focus_casing"])   # never crossed
occ_lines = occ_major | line_mask(["county"])
occ_labels = np.zeros((GH, GW), bool)

def model_number_mask(rgb, y0, y1, pad=2):
    """Boxes around the map's own numbers (contour labels, maximum): small blobs of pure black or
    pure white pixels. Long lines (frame), large text and single specks are ignored."""
    a = rgb.astype(int)[y0:y1]
    m = (a.max(2) < 60) | (a.min(2) > 248)
    lab, _ = ndi.label(ndi.binary_dilation(m, iterations=2))
    out = np.zeros(rgb.shape[:2], bool)
    for sl in ndi.find_objects(lab):
        h, w = sl[0].stop - sl[0].start, sl[1].stop - sl[1].start
        if h > 20 or w > 45 or h < 4 or sl[1].start <= 2 or sl[1].stop >= rgb.shape[1] - 2:
            continue
        out[max(0, sl[0].start + y0 - pad):sl[0].stop + y0 + pad, max(0, sl[1].start - pad):sl[1].stop + pad] = True
    return out

if PROTECT_MODEL_NUMBERS:
    nums = model_number_mask(np.asarray(img), int(box[1]), int(box[3]))
    occ_labels |= np.repeat(np.repeat(nums, S, 0), S, 1)
    print(f"Protected model numbers: {ndi.label(nums)[1]}")
# Outside the map area is never available for labels
mg = EDGE_MARGIN_PX
bx0, by0, bx1, by1 = ((box + [mg, mg, -mg, -mg]) * S).astype(int)
outside = np.ones((GH, GW), bool); outside[by0:by1, bx0:bx1] = False
occ_lines |= outside
occ_lines_tight = line_mask(["province", "iran", "foreign", "coast", "county"], NARROW_CLEARANCE) | outside
occ_major |= outside

In [ ]:
# ============================================================
# 7) Draw the map
# ============================================================
matplotlib.rcParams["pdf.fonttype"] = 42
DPI = 100 * ups
FOOTER_PX = max(20, round(H * 0.035))
HEADER_PX = round(H * 0.085) if HEADER else 0
TOTAL_PX = H + FOOTER_PX + HEADER_PX
fig = plt.figure(figsize=(W / 100, TOTAL_PX / 100), dpi=DPI)
ax = fig.add_axes([0, FOOTER_PX / TOTAL_PX, 1, H / TOTAL_PX])
ax.imshow(base_hi, extent=(0, W, H, 0), interpolation="none")
ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.axis("off")
renderer = fig.canvas.get_renderer()
clip = Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], transform=ax.transData)

for lines, style, z in LINES.values():
    if lines:
        lc = LineCollection(lines, zorder=z, capstyle="round", joinstyle="round", **style)
        ax.add_collection(lc)
        lc.set_clip_path(clip)                 # after add_collection (it resets the clip)

# --- Contrast-based text colour --------------------------------------------------------
_rgb = np.asarray(img, dtype=float) / 255.0
_lin = np.where(_rgb <= 0.04045, _rgb / 12.92, ((_rgb + 0.055) / 1.055) ** 2.4)
LUM = 0.2126 * _lin[..., 0] + 0.7152 * _lin[..., 1] + 0.0722 * _lin[..., 2]

def _lum_of(color):
    c = np.array(matplotlib.colors.to_rgb(color))
    c = np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)
    return 0.2126 * c[0] + 0.7152 * c[1] + 0.0722 * c[2]

def pick_color(L, dark):
    con = lambda a, b: (max(a, b) + 0.05) / (min(a, b) + 0.05)
    return dark if con(L, _lum_of(dark)) >= con(L, _lum_of(LIGHT_TEXT_COLOR)) else LIGHT_TEXT_COLOR

def lum_box(x0, y0, x1, y1):
    c0, c1 = int(max(0, x0)), int(min(W, x1) + 1); r0, r1 = int(max(0, y0)), int(min(H, y1) + 1)
    return LUM[r0:r1, c0:c1].mean() if (c1 > c0 and r1 > r0) else 1.0

# --- Text measurement (grid cells) -----------------------------------------------------
_size_cache = {}
def text_cells(txt, fp_key, size):
    """Width/height of a label in grid cells."""
    k = (txt, fp_key, size)
    if k not in _size_cache:
        t = ax.text(0, 0, fa(txt), fontproperties=FP[fp_key], fontsize=size, ha="center",
                    va="center", linespacing=0.95)
        e = t.get_window_extent(renderer)
        t.remove()
        _size_cache[k] = (int(np.ceil(e.width * S / ups)) + 2, int(np.ceil(e.height * S / ups)) + 2)
    return _size_cache[k]

def text_effects(color, size, fp_key, white=0.0):
    """White outline (white > 0) and, for a single-weight font, extra weight for "Black" labels."""
    heavy = SINGLE_WEIGHT and fp_key == "Black"
    fx = []
    if white > 0:
        fx.append(pe.Stroke(linewidth=white + (size * HEAVY_STROKE if heavy else 0), foreground="white"))
    if heavy:
        fx.append(pe.Stroke(linewidth=size * HEAVY_STROKE, foreground=color))
    return fx + [pe.Normal()] if fx else []

def draw_label(txt, gx, gy, fp_key, size, color, adaptive=False, z=6, stroke=0.0, pad_pt=0.6):
    """Draw a label centred on grid cell (gx, gy) and mark it occupied.
    adaptive=True switches to white over dark map colours; stroke = white outline width (pt)."""
    w, h = text_cells(txt, fp_key, size)
    x, y = gx / S, gy / S
    t = ax.text(x, y, fa(txt), fontproperties=FP[fp_key], fontsize=size, ha="center", va="center",
                linespacing=0.95, zorder=z, multialignment="center", color=color)
    fx = text_effects(color, size, fp_key, stroke)
    if fx:
        t.set_path_effects(fx)
    if adaptive:
        t.set_color(pick_color(lum_box(x - w / 2 / S, y - h / 2 / S, x + w / 2 / S, y + h / 2 / S), color))
    pad = int(pad_pt * PT * S)
    occ_labels[max(0, gy - h // 2 - pad):gy + h // 2 + pad + 1, max(0, gx - w // 2 - pad):gx + w // 2 + pad + 1] = True
    return t

def fit_centres(free, w, h):
    """Cells where a w x h box centred there lies completely in `free`."""
    return ndi.minimum_filter(free.astype(np.uint8), size=(h, w), mode="constant", cval=0) > 0

def variants(name, wrap=True):
    v = [name]
    words = name.split()
    if wrap and WRAP_TWO_WORDS and len(words) >= 2:
        cut = min(range(1, len(words)), key=lambda i: abs(len(" ".join(words[:i])) - len(" ".join(words[i:]))))
        v.append(" ".join(words[:cut]) + "\n" + " ".join(words[cut:]))
    return v

def place_free(txt, fp_key, sizes, gx0, gy0, radius, color, allow=None, adaptive=False, wrap=True,
               stroke=0.0, lines=None, centre=None, pad_pt=0.6):
    """Place a label at the free position nearest to (gx0, gy0) within radius (grid cells)."""
    r0, r1 = max(0, gy0 - radius), min(GH, gy0 + radius)
    c0, c1 = max(0, gx0 - radius), min(GW, gx0 + radius)
    lines = occ_lines if lines is None else lines
    free = ~(lines[r0:r1, c0:c1] | occ_labels[r0:r1, c0:c1])
    if allow is not None:
        free &= allow[r0:r1, c0:c1]
    cmask = None
    if centre is not None:                      # (slice, mask): label centre must lie in this mask
        csl, cm = centre
        cmask = np.zeros(free.shape, bool)
        ys0, ys1 = max(r0, csl[0].start), min(r1, csl[0].stop)
        xs0, xs1 = max(c0, csl[1].start), min(c1, csl[1].stop)
        if ys1 > ys0 and xs1 > xs0:
            cmask[ys0 - r0:ys1 - r0, xs0 - c0:xs1 - c0] = cm[ys0 - csl[0].start:ys1 - csl[0].start,
                                                          xs0 - csl[1].start:xs1 - csl[1].start]
    for size in sizes:
        for v in variants(txt, wrap):
            w, h = text_cells(v, fp_key, size)
            extra = int(np.ceil(stroke * PT * S))           # outline makes the label slightly larger
            ok = fit_centres(free, w + extra, h + extra)
            if cmask is not None:
                ok &= cmask
            if ok.any():
                yy, xx = np.nonzero(ok)
                i = np.argmin((yy + r0 - gy0) ** 2 + (xx + c0 - gx0) ** 2)
                return draw_label(v, xx[i] + c0, yy[i] + r0, fp_key, size, color, adaptive, stroke=stroke,
                                  pad_pt=pad_pt)
    return None

# --- Sea names: slanted along the water body, stretched with kashida, white outline --------
NON_JOINING = set("اآأإءدذرزژوؤةۀ ")

def kashida(name, n):
    """Insert n kashida (ـ) at the joinable position nearest the middle of each word."""
    if n <= 0:
        return name
    out = []
    for word in name.split(" "):
        cand = [i for i in range(len(word) - 1) if word[i] not in NON_JOINING and word[i + 1] != " "]
        if cand:
            i = min(cand, key=lambda k: abs(k + 0.5 - len(word) / 2))
            word = word[:i + 1] + "ـ" * n + word[i + 1:]
        out.append(word)
    return " ".join(out)

def rotated_box_mask(gx, gy, w, h, ang):
    """Grid cells covered by a w x h box centred at (gx, gy) rotated by ang degrees (screen CCW)."""
    t = np.radians(ang)
    u = np.array([np.cos(t), -np.sin(t)]); v = np.array([np.sin(t), np.cos(t)])   # y grid points down
    corners = [np.array([gx, gy]) + sx * w / 2 * u + sy * h / 2 * v for sx, sy in ((-1, -1), (1, -1), (1, 1), (-1, 1))]
    c = np.array(corners)
    x0, y0 = np.floor(c.min(0)).astype(int); x1, y1 = np.ceil(c.max(0)).astype(int)
    if x0 < 0 or y0 < 0 or x1 >= GW or y1 >= GH:
        return None
    m = Image.new("1", (x1 - x0 + 1, y1 - y0 + 1), 0)
    ImageDraw.Draw(m).polygon([tuple(p - [x0, y0]) for p in c], fill=1)
    return (slice(y0, y1 + 1), slice(x0, x1 + 1)), np.asarray(m, dtype=bool)

def place_sea(name, p1, p2, max_size):
    x1, y1 = lonlat_to_px(*p1)
    if p2 is None:
        sizes = [max_size * f for f in (1.0, 0.85, 0.7, 0.6)]
        if place_free(name, "Black", sizes, int(x1 * S), int(y1 * S), 40 * S, SEA_COLOR,
                      adaptive=ADAPTIVE_SEA_COLOR, wrap=False, stroke=max_size * SEA_STROKE) is None:
            print(f"  sea label not placed: {name}")
        return
    x2, y2 = lonlat_to_px(*p2)
    ang = -np.degrees(np.arctan2(y2 - y1, x2 - x1))          # screen angle (image y points down)
    length = np.hypot(x2 - x1, y2 - y1) * S                   # grid cells
    gx0, gy0 = (x1 + x2) / 2 * S, (y1 + y2) / 2 * S
    t = np.radians(ang); along = np.array([np.cos(t), -np.sin(t)]); perp = np.array([np.sin(t), np.cos(t)])
    blocked = occ_lines | occ_labels
    for f in (1.0, 0.9, 0.8, 0.7, 0.6, 0.5):
        size = max_size * f
        stroke = size * SEA_STROKE
        # stretch with kashida until the name covers ~85 % of the line
        txt = name
        for n in range(SEA_KASHIDA_MAX + 1):
            cand = kashida(name, n)
            if text_cells(cand, "Black", size)[0] > SEA_STRETCH * length:
                break
            txt = cand
        w, h = text_cells(txt, "Black", size)
        w += int(stroke * PT * S); h += int(stroke * PT * S)
        for da, dp in [(0, 0)] + [(a_, p_) for r_ in (1, 2, 3) for a_ in (-r_, 0, r_) for p_ in (-r_, 0, r_) if (a_, p_) != (0, 0)]:
            c = np.array([gx0, gy0]) + da * 0.06 * length * along + dp * 0.25 * h * perp
            res_ = rotated_box_mask(c[0], c[1], w, h, ang)
            if res_ is None:
                continue
            sl, m = res_
            if (blocked[sl] & m).any():
                continue
            tx = ax.text(c[0] / S, c[1] / S, fa(txt), fontproperties=FP["Black"], fontsize=size,
                         color=SEA_COLOR, ha="center", va="center", rotation=ang,
                         rotation_mode="anchor", zorder=6.8)
            tx.set_path_effects(text_effects(SEA_COLOR, size, "Black", stroke))
            occ_labels[sl] |= m
            return
    print(f"  sea label not placed (no free space along the line): {name}")

for name, p1, p2, size in P["seas"]:
    place_sea(name, p1, p2, size)
for name, lon, lat in P.get("countries", []):
    x, y = lonlat_to_px(lon, lat)
    if place_free(name, "Bold", [P["country_fontsize"], P["country_fontsize"] * 0.8], int(x * S), int(y * S),
                  60 * S, COUNTRY_COLOR) is None:
        print(f"  country name not placed: {name}")

placed_names = set()
n_inside = n_fallback = 0

# --- Province capitals: coloured star at the city + large coloured name next to it ----------
caps = counties[counties["is_capital"] & in_box(counties["x"], counties["y"])]
sym = P["capital_symbol"]
ax.scatter(caps["x"], caps["y"], s=sym ** 2, marker=CAPITAL_MARKER, facecolors=CAPITAL_STAR_COLOR,
           edgecolors="black", linewidths=0.4, zorder=6.5)
half = int(sym / 2 * PT * S) + 2
for x, y in zip(caps["x"], caps["y"]):
    gx, gy = int(x * S), int(y * S)
    occ_labels[max(0, gy - half):gy + half + 1, max(0, gx - half):gx + half + 1] = True
saved_lines = occ_lines.copy()
occ_lines[:] = outside                      # capital names may cross boundary lines, not other labels
n_cap = 0
for _, r in caps.iterrows():
    # same size for every capital; the nearest free position next to the star is used
    t = place_free(r["center_name"], "Black", [P["capital_size"]], int(r["x"] * S), int(r["y"] * S),
                   30 * S, CAPITAL_COLOR, wrap=False, stroke=CAPITAL_STROKE)
    if t is None:      # crowded or at the frame edge: search further away
        t = place_free(r["center_name"], "Black", [P["capital_size"]], int(r["x"] * S), int(r["y"] * S),
                       80 * S, CAPITAL_COLOR, wrap=False, stroke=CAPITAL_STROKE)
    if t is None:
        print(f"  capital name not placed (no room inside the frame): {r['center_name']}")
        continue
    n_cap += 1
occ_lines[:] = saved_lines
print(f"Province capitals: {n_cap}")

# --- County seats of the focus province: circle at the exact location + city name -------------
def seat_display_name(n):
    """'بندرکنگان' -> 'بندر کنگان' (the source list writes some port names without a space)."""
    n = str(n)
    return "بندر " + n[4:] if n.startswith("بندر") and len(n) > 4 and n[4] != " " else n

if SHOW_FOCUS_SEATS and P.get("focus_province"):
    _fc = capitals[capitals["province_name"] == P["focus_province"]]
    if len(_fc):
        seats = counties[(counties["province_id"] == int(_fc["province_id"].iloc[0]))
                         & ~counties["is_capital"] & in_box(counties["x"], counties["y"])]
        ax.scatter(seats["x"], seats["y"], s=SEAT_MARKER ** 2, marker="o", facecolors=SEAT_FACE,
                   edgecolors=SEAT_EDGE, linewidths=0.6, zorder=6.4)
        hc = int(SEAT_MARKER / 2 * PT * S) + 2
        for x, y in zip(seats["x"], seats["y"]):            # circles are obstacles for every label
            gx, gy = int(x * S), int(y * S)
            occ_labels[max(0, gy - hc):gy + hc + 1, max(0, gx - hc):gx + hc + 1] = True
        coast_dist = ndi.distance_transform_edt(~line_mask(["coast"], 0.0))
        n_seat = 0
        for _, r in seats.iterrows():
            nm = seat_display_name(r["center_name"])
            gx, gy = int(r["x"] * S), int(r["y"] * S)
            # next to the circle, not across the coast / province lines; else crossing any line
            # 1) on land, not across coast / province lines  2) anywhere clear of those lines
            # 3) crossing lines if nothing else fits; a wider gap keeps neighbouring seat names apart
            # Coastal seats: directly next to the circle, crossing the coast / over the sea is fine
            t = None
            coastal = bool(coast_dist[min(gy, GH - 1), min(gx, GW - 1)] <= COASTAL_SEAT_PX * S)
            order_ = (((None, outside),) if coastal else
                      ((county_grid > 0, occ_major), (None, occ_major), (None, outside)))
            for allow_, lines_ in order_:
                t = place_free(nm, "Regular", SEAT_SIZES, gx, gy, 28 * S, SEAT_COLOR, wrap=False,
                               lines=lines_, allow=allow_, pad_pt=1.5)
                if t is not None:
                    break
            if t is None:
                print(f"  seat name not placed: {nm}")
                continue
            n_seat += 1
        print(f"County seats of {P['focus_province']}: {len(seats)} circles, {n_seat} names")

if P["label_mode"] == "area":
    # --- Which seats belong to which polygon --------------------------------------------
    names_in = {k: [] for k in range(len(adm2))}
    for _, r in counties.iterrows():
        k = int(r["adm2"])
        if k == -1 and in_box(r["x"], r["y"]):
            # Seat just outside every polygon (e.g. on the shoreline): use the nearest county cell
            gx, gy = int(r["x"] * S), int(r["y"] * S)
            win_ = county_grid[max(0, gy - 6 * S):gy + 6 * S, max(0, gx - 6 * S):gx + 6 * S]
            if (win_ > 0).any():
                yy, xx = np.nonzero(win_)
                j = np.argmin((yy - min(gy, 6 * S)) ** 2 + (xx - min(gx, 6 * S)) ** 2)
                k = int(win_[yy[j], xx[j]]) - 1
        if k >= 0:
            names_in[k].append(r)
    for k, (pname, _) in enumerate(adm2):
        if pname in POLYGON_NAME_FIXES and not names_in[k]:
            names_in[k] = [pd.Series(dict(county_name=POLYGON_NAME_FIXES[pname], x=np.nan, y=np.nan,
                                           is_capital=False))]
    focus_pid = None
    if P.get("focus_province"):
        _cap = capitals[capitals["province_name"] == P["focus_province"]]
        focus_pid = int(_cap["province_id"].iloc[0]) if len(_cap) else None
    slices = ndi.find_objects(county_grid)
    in_map = ~outside
    jobs = []      # (priority, mask slice, sub-mask, name, is_capital, pole)
    for k, sl in enumerate(slices):
        if sl is None or not names_in.get(k):
            continue
        region = county_grid[sl] == (k + 1)
        seats = names_in[k]
        if len(seats) == 1:
            parts = [(region, seats[0])]
        else:
            # New counties inside an old polygon: split the polygon between the seats (nearest seat)
            yy, xx = np.mgrid[sl[0], sl[1]]
            d = np.stack([(xx - s["x"] * S) ** 2 + (yy - s["y"] * S) ** 2 for s in seats])
            owner = np.argmin(d, axis=0)
            parts = [(region & (owner == j), s) for j, s in enumerate(seats)]
        for sub, s in parts:
            sub = sub & in_map[sl]
            if sub.sum() < 40 * S * S:       # county (part) hardly visible in this frame: no label
                continue
            if s["is_capital"] and s.get("center_name") == s["county_name"] and in_box(s["x"], s["y"]):
                continue                     # already named by the province-capital label
            foc = bool(focus_pid is not None and s.get("province_id") == focus_pid)
            jobs.append((not foc, -sub.sum(), sl, sub, s["county_name"], foc))
    jobs.sort(key=lambda j: (j[0], j[1]))

    # focus-province counties: heaviest font, largest sizes; all others: lighter and smaller
    if focus_pid is None:
        fp_of = lambda foc: "Bold"
        sizes_of = lambda foc: P["county_sizes"]
    else:
        fp_of = lambda foc: "Black" if foc else "Bold"
        sizes_of = lambda foc: P["focus_sizes"] if foc else P["county_sizes"]
    pending = []
    def long_axis_angle(region):
        """Screen angle (deg, counter-clockwise) of the long axis of a region; 0 if roundish."""
        yy, xx = np.nonzero(region)
        if len(xx) < 10:
            return 0.0
        cov = np.cov(np.vstack([xx - xx.mean(), yy - yy.mean()]))
        ev, evec = np.linalg.eigh(cov)
        if ev[1] < 1.4 * max(ev[0], 1e-9):        # not elongated enough to justify a slant
            return 0.0
        ex, ey = evec[:, 1]
        ang = -np.degrees(np.arctan2(ey, ex))
        ang = (ang + 90) % 180 - 90                 # readable direction: (-90, 90]
        return float(np.clip(ang, -MAX_SLANT, MAX_SLANT))

    def rotate_window(mask, ang):
        """Resample a mask into the text frame of angle ang; returns (rotated mask, mapper back)."""
        h0, w0 = mask.shape
        D = int(np.ceil(np.hypot(h0, w0))) + 2
        t = np.radians(ang)
        u = np.array([np.cos(t), -np.sin(t)]); v = np.array([np.sin(t), np.cos(t)])
        cx, cy = (w0 - 1) / 2, (h0 - 1) / 2
        r, c = np.mgrid[0:D, 0:D]
        dx, dy = c - D / 2, r - D / 2
        X = cx + dx * u[0] + dy * v[0]; Y = cy + dx * u[1] + dy * v[1]
        rot = ndi.map_coordinates(mask.astype(np.float32), [Y, X], order=0, cval=0) > 0.5
        back = lambda rr, cc: (cx + (cc - D / 2) * u[0] + (rr - D / 2) * v[0],
                               cy + (cc - D / 2) * u[1] + (rr - D / 2) * v[1])
        return rot, back

    for _, _, sl, sub, name, cap in jobs:
        free = sub & ~occ_lines[sl] & ~occ_labels[sl]
        if not free.any():
            pending.append((sl, sub, name, cap)); continue
        angles = [0.0]
        if SLANTED_COUNTY_NAMES and (focus_pid is None or cap):
            a = long_axis_angle(sub)
            if abs(a) >= 5:
                angles = [a, 0.0]                   # slanted first, horizontal as fallback
        frames = []
        for a in angles:
            if a == 0.0:
                frames.append((a, free, None))
            else:
                rot, back = rotate_window(free, a)
                frames.append((a, rot, back))
        done = False
        inside_sizes = [z_ for z_ in sizes_of(cap) if not (cap and focus_pid is not None and z_ < FOCUS_MIN_INSIDE)]
        for size in inside_sizes:
            for v in variants(name):
                w, h = text_cells(v, fp_of(cap), size)
                for a, fm_, back in frames:
                    ok = fit_centres(fm_, w, h)
                    if not ok.any():
                        continue
                    sc = np.where(ok, ndi.distance_transform_edt(fm_), -1)
                    yy, xx = np.unravel_index(np.argmax(sc), sc.shape)
                    if back is None:
                        draw_label(v, xx + sl[1].start, yy + sl[0].start, fp_of(cap), size, TEXT_COLOR)
                    else:
                        wx, wy = back(yy, xx)
                        gx, gy = wx + sl[1].start, wy + sl[0].start
                        tr = ax.text(gx / S, gy / S, fa(v), fontproperties=FP[fp_of(cap)], fontsize=size,
                                     color=TEXT_COLOR, ha="center", va="center", rotation=a,
                                     rotation_mode="anchor", linespacing=0.95, multialignment="center",
                                     zorder=6)
                        fx = text_effects(TEXT_COLOR, size, fp_of(cap))
                        if fx:
                            tr.set_path_effects(fx)
                        pad = int(0.6 * PT * S)
                        res_ = rotated_box_mask(gx, gy, w + 2 * pad, h + 2 * pad, a)
                        if res_ is not None:
                            occ_labels[res_[0]] |= res_[1]
                    placed_names.add(name); n_inside += 1; done = True
                    break
                if done:
                    break
            if done:
                break
        if not done and cap and focus_pid is not None and SLANTED_COUNTY_NAMES:
            # narrow focus county: along its long axis, smaller size, tighter clearance
            a = long_axis_angle(sub)
            free_n = sub & ~occ_lines_tight[sl] & ~occ_labels[sl]
            if free_n.any():
                rot, back = rotate_window(free_n, a) if abs(a) >= 1 else (free_n, None)
                for size in NARROW_SIZES:
                    w, h = text_cells(name, fp_of(cap), size)
                    ok = fit_centres(rot, w, h)
                    if not ok.any():
                        continue
                    sc = np.where(ok, ndi.distance_transform_edt(rot), -1)
                    yy, xx = np.unravel_index(np.argmax(sc), sc.shape)
                    if back is None:
                        gx, gy = xx + sl[1].start, yy + sl[0].start
                    else:
                        wx, wy = back(yy, xx)
                        gx, gy = wx + sl[1].start, wy + sl[0].start
                    tr = ax.text(gx / S, gy / S, fa(name), fontproperties=FP[fp_of(cap)], fontsize=size,
                                 color=TEXT_COLOR, ha="center", va="center", rotation=a if back else 0,
                                 rotation_mode="anchor", zorder=6)
                    fx = text_effects(TEXT_COLOR, size, fp_of(cap))
                    if fx:
                        tr.set_path_effects(fx)
                    res_ = rotated_box_mask(gx, gy, w + 2, h + 2, a if back else 0)
                    if res_ is not None:
                        occ_labels[res_[0]] |= res_[1]
                    placed_names.add(name); n_inside += 1; done = True
                    print(f"  narrow county written along its strip: {name} ({size} pt, {a:.0f} deg)")
                    break
        if not done:
            pending.append((sl, sub, name, cap))
    # Counties too small for their name:
    #  1) nearest free spot around the county centre, crossing only thin county lines (no leader line)
    #  2) otherwise (only if the preset allows leader lines) the nearest spot clear of all lines,
    #     connected by a thin leader line; else the name is skipped
    pending.sort(key=lambda j: j[1].sum())
    land_grid = county_grid > 0
    n_dropped = 0
    for sl, sub, name, cap in pending:
        cen = ndi.distance_transform_edt(sub)
        py, px = np.unravel_index(np.argmax(cen), cen.shape)
        gx, gy = px + sl[1].start, py + sl[0].start
        is_focus = bool(cap and focus_pid is not None)
        fb_sizes = sizes_of(cap) if is_focus else sizes_of(cap)[-2:]
        rad_ = (18 if is_focus else 10) * S
        # 1) centre inside the own county, label on land, may cross only thin county lines
        t = place_free(name, fp_of(cap), fb_sizes, gx, gy, rad_, TEXT_COLOR, lines=occ_major,
                       allow=land_grid, centre=(sl, sub))
        # 2) nearest spot on land around the county (still no leader line)
        if t is None:
            t = place_free(name, fp_of(cap), fb_sizes, gx, gy, rad_, TEXT_COLOR, lines=occ_major,
                           allow=land_grid)
        if t is None and not P.get("leader_lines", True):
            n_dropped += 1                    # whole-country frames: skip rather than clutter
            continue
        if t is None:
            t = place_free(name, fp_of(cap), fb_sizes, gx, gy, 40 * S, TEXT_COLOR)
        if t is None:
            t = draw_label(name, gx, gy, fp_of(cap), sizes_of(cap)[-1], TEXT_COLOR)
        tx, ty = t.get_position()
        if np.hypot(tx - gx / S, ty - gy / S) > 7:
            ax.plot([gx / S, tx], [gy / S, ty], color="#333333", lw=0.3, zorder=5.5)
            ax.plot(gx / S, gy / S, "o", ms=0.9, color="#333333", zorder=5.6)
        placed_names.add(name); n_fallback += 1
    if SHOW_SEATS_AREA:
        sel = counties[in_box(counties["x"], counties["y"])]
        ax.scatter(sel["x"], sel["y"], s=1.2, marker="s", facecolors="none", edgecolors="#222222",
                   linewidths=0.3, zorder=5)
    print(f"Area labels: {n_inside} inside their county, {n_fallback} next to it (small county), "
          f"{n_dropped} skipped (no room at a readable size)")

else:
    # --- Point mode: hollow squares at county seats, labels around them -----------------
    sel = counties[in_box(counties["x"], counties["y"]) & ~counties["is_capital"]].copy()
    side = P.get("county_marker", 1.5)
    ax.scatter(sel["x"], sel["y"], s=side ** 2, marker="s", facecolors="none", edgecolors=TEXT_COLOR,
               linewidths=0.35, zorder=4.5)
    for x, y in zip(sel["x"], sel["y"]):
        hc = int(side / 2 * PT * S) + 1
        occ_labels[int(y * S) - hc:int(y * S) + hc + 1, int(x * S) - hc:int(x * S) + hc + 1] = True
    occ_lines_saved = occ_lines.copy()
    occ_lines[:] = outside                     # point labels may cross boundary lines
    for _, r in sel.iterrows():
        name, size = r["county_name"], P.get("county_fontsize", P["county_sizes"][0])
        t = place_free(name, "Bold", [size], int(r["x"] * S), int(r["y"] * S), 12 * S, TEXT_COLOR)
        if t is None:
            draw_label(name, int(r["x"] * S), int(r["y"] * S) - 3 * S, "Bold", size, TEXT_COLOR)
            n_fallback += 1
        else:
            n_inside += 1
        placed_names.add(r["county_name"])
    occ_lines[:] = occ_lines_saved
    print(f"Point labels: {n_inside} without overlap, {n_fallback} overlapping")

# --- Islands ------------------------------------------------------------------------------
water = county_grid == 0
n_isl = 0
for name, lon, lat in ISLANDS:
    x, y = lonlat_to_px(lon, lat)
    if not in_box(x, y) or name in placed_names:
        continue
    t = place_free(ISLAND_PREFIX + name, "Bold", [P["island_fontsize"], P["island_fontsize"] * 0.85],
                   int(x * S), int(y * S), 22 * S, TEXT_COLOR, allow=water)
    n_isl += t is not None
print(f"Island labels: {n_isl}")

# --- Safety net: remove any label that does not lie completely inside the map area ----------
inner = ax.transData.transform([(box[0] + 1, box[3] - 1), (box[2] - 1, box[1] + 1)])
n_cut = 0
for t in list(ax.texts):
    e = t.get_window_extent(renderer)
    if e.x0 < inner[0][0] or e.x1 > inner[1][0] or e.y0 < inner[0][1] or e.y1 > inner[1][1]:
        t.remove(); n_cut += 1
if n_cut:
    print(f"Removed {n_cut} label(s) that would have been cut by the frame")

# --- Header box: model and forecast period (Persian calendar, Iran time) ------------------------
from datetime import datetime, timedelta
from matplotlib.patches import FancyBboxPatch

def to_jalali(gy, gm, gd):
    """Gregorian -> Persian (Jalali) calendar date."""
    g_d_m = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]
    gy2 = gy + 1 if gm > 2 else gy
    days = (355666 + 365 * gy + (gy2 + 3) // 4 - (gy2 + 99) // 100 + (gy2 + 399) // 400
            + gd + g_d_m[gm - 1])
    jy = -1595 + 33 * (days // 12053); days %= 12053
    jy += 4 * (days // 1461); days %= 1461
    if days > 365:
        jy += (days - 1) // 365; days = (days - 1) % 365
    if days < 186:
        return jy, 1 + days // 31, 1 + days % 31
    return jy, 7 + (days - 186) // 30, 1 + (days - 186) % 30

# Persian digits if the font has them; older "B" fonts draw the ASCII digits with Persian shapes
FA_DIGITS = (str.maketrans("0123456789", "۰۱۲۳۴۵۶۷۸۹") if 0x06F0 in FONT_CMAP
             else str.maketrans("", ""))
FA_MONTHS = ["فروردین", "اردیبهشت", "خرداد", "تیر", "مرداد", "شهریور",
             "مهر", "آبان", "آذر", "دی", "بهمن", "اسفند"]
FA_WEEKDAYS = ["دوشنبه", "سه‌شنبه", "چهارشنبه", "پنجشنبه", "جمعه", "شنبه", "یکشنبه"]   # Monday first

def iran_time(utc_text):
    """'2026-11-06 06:00' (UTC) -> ('۰۹:۳۰', 'جمعه ۱۵ آبان ۱۴۰۵') in Iran time."""
    t = datetime.strptime(utc_text.strip(), "%Y-%m-%d %H:%M") + timedelta(hours=IRAN_UTC_OFFSET[0],
                                                                         minutes=IRAN_UTC_OFFSET[1])
    jy, jm, jd = to_jalali(t.year, t.month, t.day)
    return (t.strftime("%H:%M").translate(FA_DIGITS),
            f"{FA_WEEKDAYS[t.weekday()]} {str(jd).translate(FA_DIGITS)} {FA_MONTHS[jm - 1]} "
            f"{str(jy).translate(FA_DIGITS)}")

if HEADER:
    t1, d1 = iran_time(HEADER["start_utc"])
    t2, d2 = iran_time(HEADER["end_utc"])
    line1 = HEADER["title"].format(model=HEADER["model"])
    line2 = HEADER["period"].format(t1=t1, d1=d1, t2=t2, d2=d2)
    hb = HEADER_PX / TOTAL_PX                     # header height (figure fraction)
    y0 = 1 - hb
    mx, my = 6 / W, 0.10 * hb                     # outer margins
    fig.patches.append(FancyBboxPatch((mx, y0 + my), 1 - 2 * mx, hb - 2 * my,
                                      boxstyle="round,pad=0,rounding_size=0.006",
                                      transform=fig.transFigure, facecolor=HEADER_FILL,
                                      edgecolor="none", linewidth=0, zorder=1))
    size1 = HEADER_PX * 0.30 * 72 / 100           # font sizes scale with the header height
    size2 = HEADER_PX * 0.20 * 72 / 100
    h1 = fig.text(0.5, y0 + hb * 0.64, fa(line1), ha="center", va="center", fontproperties=FP["Black"],
                  fontsize=size1, color=HEADER_COLOR, zorder=2)
    h2 = fig.text(0.5, y0 + hb * 0.30, fa(line2), ha="center", va="center", fontproperties=FP["Bold"],
                  fontsize=size2, color="#222222", zorder=2)
    for t_, sz in ((h1, size1), (h2, size2)):
        fx = text_effects(t_.get_color(), sz, "Black")
        if fx:
            t_.set_path_effects(fx)
    print(f"Header: {line1} | {line2}")

# --- Credit line (and optional second line) ---------------------------------------------------
fh = FOOTER_PX / TOTAL_PX
fig.text(0.5, fh * (0.62 if SOURCE_NOTE else 0.5), CREDIT_TEXT, ha="center", va="center",
         fontsize=P["credit_fontsize"], color="#222222", fontweight="bold")
if SOURCE_NOTE:
    fig.text(0.5, fh * 0.22, SOURCE_NOTE, ha="center", va="center", fontsize=P["note_fontsize"],
             color="#666666")

fig.savefig(OUTPUT_IMAGE, dpi=DPI, facecolor="white")
print(f"Saved: {OUTPUT_IMAGE}  ({W * ups} x {TOTAL_PX * ups} px)")
if OUTPUT_PDF:
    fig.savefig(OUTPUT_PDF, dpi=DPI, facecolor="white")
    print(f"Saved: {OUTPUT_PDF}  (vector labels and lines)")
plt.close(fig)

In [ ]:
# ============================================================
# 8) Preview (open the PNG/PDF to zoom in)
# ============================================================
from IPython.display import Image as IPyImage, display
display(IPyImage(filename=OUTPUT_IMAGE, width=1000))